In [ ]:
import requests

response = requests.get(
    "https://api.frankfurter.dev/v1/latest",
    params={"base": "USD", "symbols": "EUR,GBP,PKR"}
)
print(response.status_code)
print(response.json())

## Extraction
Pulling daily exchange rates for USD → EUR, GBP, PKR from the Frankfurter API.
No transformation here — raw responses only, saved as-is for auditability.

In [ ]:
def fetch_rates(target_date="latest", base_currency="USD", symbols=("EUR", "GBP", "PKR")):
    url = f"https://api.frankfurter.dev/v1/{target_date}"
    params = {"base": base_currency, "symbols": ",".join(symbols)}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()  # raises an error if the request failed
    return response.json()

data = fetch_rates()
data

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

def save_raw_response(payload, target_date="latest"):
    rate_date = payload.get("date", target_date)
    collected_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    filename = f"{rate_date}_{collected_at}.json"
    filepath = RAW_DATA_DIR / filename

    record = {
        "collected_at_utc": collected_at,
        "requested_date": target_date,
        "source": "frankfurter.dev",
        "payload": payload,
    }

    with open(filepath, "w") as f:
        json.dump(record, f, indent=2)

    print(f"Saved to {filepath}")
    return filepath

save_raw_response(data)

In [ ]:
save_raw_response(fetch_rates())

In [ ]:
def validate_payload(payload):
    assert "rates" in payload, "Missing 'rates' key"
    assert "date" in payload, "Missing 'date' key"
    assert len(payload["rates"]) > 0, "Empty rates dict"
    print("Payload looks valid ✓")

validate_payload(data)

In [ ]:
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

MAX_RETRIES = 3
BACKOFF_SECONDS = 2

class ExtractionError(Exception):
    """Raised when the API can't be reached after all retries."""
    pass

def fetch_rates_with_retry(target_date="latest", base_currency="USD", symbols=("EUR", "GBP", "PKR")):
    url = f"https://api.frankfurter.dev/v1/{target_date}"
    params = {"base": base_currency, "symbols": ",".join(symbols)}

    last_exception = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.info(f"Fetching rates (attempt {attempt}/{MAX_RETRIES})")
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            payload = response.json()

            if "rates" not in payload:
                raise ExtractionError(f"Unexpected response shape: {payload}")

            return payload

        except (requests.RequestException, ExtractionError) as exc:
            last_exception = exc
            logger.warning(f"Attempt {attempt} failed: {exc}")
            if attempt < MAX_RETRIES:
                sleep_time = BACKOFF_SECONDS * attempt
                logger.info(f"Retrying in {sleep_time}s...")
                time.sleep(sleep_time)

    raise ExtractionError(f"Failed after {MAX_RETRIES} attempts") from last_exception

In [ ]:
data = fetch_rates_with_retry()
data

In [ ]:
# Deliberately call a bad endpoint to trigger retries and confirm backoff behavior
try:
    fetch_rates_with_retry(target_date="not-a-real-date")
except ExtractionError as e:
    print(f"\nCaught expected failure: {e}")

In [ ]:
import json
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")

def load_raw_files():
    records = []
    for filepath in sorted(RAW_DATA_DIR.glob("*.json")):
        with open(filepath) as f:
            records.append(json.load(f))
    return records

raw_records = load_raw_files()
print(f"Loaded {len(raw_records)} raw files")
raw_records[0]

In [ ]:
def flatten_record(record):
    """
    Turn one raw record into a list of clean rows — one row per currency pair.
    e.g. one API call covering EUR, GBP, PKR becomes 3 rows.
    """
    payload = record["payload"]
    rate_date = payload["date"]
    base = payload["base"]
    collected_at = record["collected_at_utc"]

    rows = []
    for currency, rate in payload["rates"].items():
        rows.append({
            "date": rate_date,
            "base_currency": base,
            "target_currency": currency,
            "rate": rate,
            "collected_at_utc": collected_at,
        })
    return rows

# test on the first record only
test_rows = flatten_record(raw_records[0])
test_rows

In [ ]:
import pandas as pd

EXPECTED_CURRENCIES = {"EUR", "GBP", "PKR"}

def flatten_all_records(records):
    all_rows = []
    missing_log = []

    for record in records:
        rows = flatten_record(record)
        present_currencies = {row["target_currency"] for row in rows}
        missing = EXPECTED_CURRENCIES - present_currencies

        if missing:
            missing_log.append({
                "date": record["payload"]["date"],
                "missing_currencies": sorted(missing),
            })

        all_rows.extend(rows)

    return pd.DataFrame(all_rows), missing_log

df, missing_log = flatten_all_records(raw_records)
df

In [ ]:
missing_log

### Handling missing currencies

Frankfurter's free tier occasionally omits PKR from multi-currency responses
(observed on 2026-07-07 across multiple collections). Rather than forward-filling
or estimating a rate, missing currency-date pairs are simply left absent from
the dataset and logged separately in `missing_log`. This keeps the dataset
honest — every row reflects an actual API-confirmed rate, not an assumption.

In [ ]:
def add_pct_change(df):
    df = df.sort_values(["target_currency", "date"]).copy()
    df["pct_change"] = df.groupby("target_currency")["rate"].pct_change() * 100
    return df

df = add_pct_change(df)
df

In [ ]:
def dedupe_by_date(df):
    """
    Keep only the latest collection per (date, currency) pair, so
    pct_change reflects true day-over-day movement, not repeated
    same-day collections during testing/backfills.
    """
    df = df.sort_values("collected_at_utc")
    df = df.drop_duplicates(subset=["date", "target_currency"], keep="last")
    return df.sort_values(["target_currency", "date"]).reset_index(drop=True)

df = dedupe_by_date(df)
df = add_pct_change(df)
df

In [ ]:
import sqlite3

DB_PATH = Path("data/currency_rates.db")

def load_to_sqlite(df, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS exchange_rates (
            date TEXT NOT NULL,
            base_currency TEXT NOT NULL,
            target_currency TEXT NOT NULL,
            rate REAL NOT NULL,
            pct_change REAL,
            collected_at_utc TEXT NOT NULL,
            PRIMARY KEY (date, base_currency, target_currency)
        )
    """)

    # upsert: avoids duplicate rows if this runs more than once for the same date
    df.to_sql("exchange_rates_staging", conn, if_exists="replace", index=False)
    conn.execute("""
        INSERT OR REPLACE INTO exchange_rates
        SELECT date, base_currency, target_currency, rate, pct_change, collected_at_utc
        FROM exchange_rates_staging
    """)
    conn.execute("DROP TABLE exchange_rates_staging")
    conn.commit()
    conn.close()
    print(f"Loaded {len(df)} rows into {db_path}")

load_to_sqlite(df)

In [ ]:
load_to_sqlite(df)

In [ ]:
conn = sqlite3.connect(DB_PATH)
result = pd.read_sql("SELECT * FROM exchange_rates ORDER BY date, target_currency", conn)
conn.close()
result

In [ ]:
# check what's actually in the dataframe before it hit SQLite
df[["target_currency", "rate", "pct_change"]]

In [ ]:
def run_pipeline():
    """
    Full ETL run: extract latest rates, save raw, transform all
    historical raw files, and load into SQLite.
    """
    # Extract
    payload = fetch_rates_with_retry()
    save_raw_response(payload, target_date="latest")

    # Transform (reprocesses all raw files, so the table always
    # reflects the complete history, not just today's new file)
    raw_records = load_raw_files()
    df, missing_log = flatten_all_records(raw_records)
    df = dedupe_by_date(df)
    df = add_pct_change(df)

    if missing_log:
        print(f"⚠ Missing currencies detected: {missing_log}")

    # Load
    load_to_sqlite(df)

    return df

result_df = run_pipeline()
result_df

In [ ]:
from src.extract import fetch_rates_with_retry, save_raw_response, load_raw_files
from src.transform_load import flatten_all_records, dedupe_by_date, add_pct_change, load_to_sqlite

def run_pipeline():
    payload = fetch_rates_with_retry()
    save_raw_response(payload, target_date="latest")

    raw_records = load_raw_files()
    df, missing_log = flatten_all_records(raw_records)
    df = dedupe_by_date(df)
    df = add_pct_change(df)

    if missing_log:
        print(f"⚠ Missing currencies detected: {missing_log}")

    load_to_sqlite(df)
    return df

result_df = run_pipeline()
result_df

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("data/currency_rates.db")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM exchange_rates ORDER BY target_currency, date", conn)
conn.close()

df

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

In [ ]:
import matplotlib.pyplot as plt

import matplotlib.dates as mdates


fig, ax = plt.subplots(figsize=(10, 5))

for currency in df["target_currency"].unique():
    subset = df[df["target_currency"] == currency]
    ax.plot(subset["date"], subset["rate"], marker="o", label=f"USD/{currency}")

ax.set_title("USD Exchange Rate Trend")
ax.set_xlabel("Date")
ax.set_ylabel("Exchange Rate")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("data/rate_trend.png", dpi=150)
plt.show()

In [ ]:
threshold = 1.0
volatile_days = df[df["pct_change"].abs() > threshold]

if not volatile_days.empty:
    print("Volatility alerts (>1% single-day move):")
    print(volatile_days[["date", "target_currency", "pct_change"]])
else:
    print("No volatility alerts — all moves within ±1% threshold.")

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(10, 4))

currencies = df["target_currency"].unique()
bar_width = 0.35  # narrower bars

for i, currency in enumerate(currencies):
    subset = df[df["target_currency"] == currency].dropna(subset=["pct_change"])
    # offset each currency's bars slightly so they sit side-by-side, not stacked
    offset = pd.Timedelta(hours=(i - len(currencies)/2) * 8)
    ax.bar(subset["date"] + offset, subset["pct_change"], label=f"USD/{currency}", alpha=0.8, width=0.3)

ax.axhline(y=1.0, color="red", linestyle="--", linewidth=1, label="Volatility threshold (±1%)")
ax.axhline(y=-1.0, color="red", linestyle="--", linewidth=1)
ax.axhline(y=0, color="black", linewidth=0.8)

ax.set_title("Daily % Change vs Volatility Threshold")
ax.set_xlabel("Date")
ax.set_ylabel("% Change")
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("data/volatility_chart.png", dpi=150)
plt.show()